# Hyperparameter Tuning — Grid Search, then Bayesian Search

How the hyperparameters in `src/models.py` (`FINAL_PARAMS`) were found (log: `reports/final_hyperparameters.json`).

1. **Grid search** over a coarse grid per model.
2. **Bayesian search** (Optuna TPE) in a narrower space centred on the best grid point; the grid best is its first trial.
3. Both minimise **5-fold CV RMSE on the seed-42 training split** (preprocessing re-fitted inside every fold;
   validation and test are never seen).
4. Library defaults, grid-best and Bayesian-best are then scored on seeds 42, 43, 44 and the version with the
   **lowest mean test RMSE** is kept.

Linear Regression and the baselines have no hyperparameters. Re-running takes about 20–40 minutes and only displays
results; it does not change `FINAL_PARAMS`.

## 1. Setup

In [ ]:
import sys, pathlib, warnings
ROOT = next((p for p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents) if (p / 'src' / 'config.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the project folder (the one that contains src/).')
sys.path.insert(0, str(ROOT))   # project root -> `import src`, whatever the working folder
warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.float_format', '{:,.2f}'.format)
from src import config
from src.preprocessing import prepare_data
from src.models import display_name, GRID_PARAMS, BAYES_PARAMS, FINAL_PARAMS
from src.tuning import TUNABLE, GRIDS, tune, compare_on_seeds

## 2. Search grids

In [ ]:
pd.DataFrame({'grid searched': {display_name(k): GRIDS[k] for k in TUNABLE}})

## 3. Grid search → Bayesian search (seed-42 training split, 5-fold CV RMSE)

In [ ]:
data = prepare_data(seed=config.RANDOM_STATE)
tuned = {key: tune(key, data) for key in TUNABLE}

## 4. Library defaults vs grid-best vs Bayesian-best — means over seeds 42, 43, 44

In [ ]:
tables = [compare_on_seeds(k, {'default': {}, 'grid': tuned[k]['grid']['params'],
                               'bayesian': tuned[k]['bayes']['params']}) for k in TUNABLE]
comparison = pd.concat(tables, ignore_index=True)
comparison['model'] = comparison['model'].map(display_name)
comparison['kept'] = comparison.groupby('model')['test_rmse'].transform('min').eq(comparison['test_rmse'])
comparison

The search results recorded in `src/models.py` (`GRID_PARAMS`, `BAYES_PARAMS`) are from the original run; on another
machine a re-run can differ slightly for models with random subsampling.